In [ ]:
# ===============================================================
# BoTorch Bayesian Optimization with a Regression Tree surrogate
# - Initial data from X_init.npy (n, 2) and y_init.npy (n or n,1)
# - Tree surrogate = bootstrapped ensemble of DecisionTreeRegressor
# - MC Expected Improvement acquisition
# - Discrete candidate search (Sobol) for robustness with trees
# - Convergence plot of incumbent best
# ===============================================================

import os
import numpy as np
import torch
from torch import Tensor
from torch.distributions import Normal, Independent

from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import resample

import matplotlib.pyplot as plt

# BoTorch bits
from botorch.models.model import Model
from botorch.posteriors.torch import TorchPosterior
from botorch.acquisition.monte_carlo import qExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.sampling import draw_sobol_samples

# -------------------------
# 0) Utilities: normalization
# -------------------------
def compute_affine_bounds(X: np.ndarray, pad: float = 0.05):
    """Return raw min/max with a small padding; handles degenerate ranges."""
    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    spans = np.maximum(maxs - mins, 1e-9)
    mins = mins - pad * spans
    maxs = maxs + pad * spans
    return mins, maxs

def normalize(X_raw: np.ndarray, mins: np.ndarray, maxs: np.ndarray) -> np.ndarray:
    return (X_raw - mins) / (maxs - mins + 1e-12)

def denormalize(X_unit: np.ndarray, mins: np.ndarray, maxs: np.ndarray) -> np.ndarray:
    return X_unit * (maxs - mins) + mins

# -------------------------------------------------
# 1) Surrogate: Bootstrapped ensemble of CART trees
# -------------------------------------------------
class BootstrappedCARTModel(Model):
    """
    A simple BoTorch-compatible surrogate built from an ensemble of CART trees.
    The ensemble mean gives μ(x); the sample std across trees gives σ(x).
    We return a TorchPosterior as an (Independent) Normal, enabling MC-EI.
    """
    _num_outputs = 1  # single-output regression

    def __init__(self, trees, x_mins: np.ndarray, x_maxs: np.ndarray, device=None, dtype=torch.double):
        """
        trees: list[DecisionTreeRegressor] fitted on normalized X in [0,1]^d
        x_mins, x_maxs: raw-space bounds (to help with pretty-printing if needed)
        """
        super().__init__()
        self.trees = trees
        self.register_buffer("x_mins", torch.tensor(x_mins, dtype=dtype))
        self.register_buffer("x_maxs", torch.tensor(x_maxs, dtype=dtype))
        self.to(device=device, dtype=dtype)

    @property
    def num_outputs(self) -> int:
        return self._num_outputs

    def posterior(self, X: Tensor, output_indices=None, observation_noise=False, posterior_transform=None, **kwargs):
        """
        X: Tensor of shape [b, q, d] (d=2 here)
        Returns TorchPosterior with rsample().
        """
        self.eval()
        X = self.transform_inputs(X)                 # no-op unless transforms were added
        b, q, d = X.shape
        X_flat = X.reshape(-1, d).detach().cpu().numpy()  # normalized space expected

        # Predict with each tree -> [n_points]
        preds = []
        for t in self.trees:
            y = t.predict(X_flat)  # numpy float array (n_points,)
            preds.append(y)
        preds = np.stack(preds, axis=0)              # [n_trees, n_points]
        mu = preds.mean(axis=0)                      # [n_points]
        std = preds.std(axis=0, ddof=1)              # [n_points]
        std = np.maximum(std, 1e-6)                  # numerical floor

        # Convert to tensors, shape to [b, q, m=1]
        mu_t = torch.tensor(mu, dtype=X.dtype, device=X.device).reshape(b, q, 1)
        std_t = torch.tensor(std, dtype=X.dtype, device=X.device).reshape(b, q, 1)

        base = Independent(Normal(loc=mu_t, scale=std_t), 1)  # event dim = 1 (m=1)
        post = TorchPosterior(base)
        if posterior_transform is not None:
            post = posterior_transform(post)
        return post

# ---------------------------------------------------
# 2) Build (fit) the regression-tree ensemble surrogate
# ---------------------------------------------------
def fit_bootstrapped_cart(
    X_unit: np.ndarray, y: np.ndarray,
    n_trees: int = 50,
    # Explicit hyperparameters:
    max_depth: int = 5,
    min_samples_split: int = 4,
    min_samples_leaf: int = 2,
    random_state: int = 7,
):
    """
    Fit n_trees bootstrap CARTs on normalized X in [0,1]^d. Returns list of trees.
    """
    rng = np.random.RandomState(random_state)
    trees = []
    y = y.reshape(-1)

    for i in range(n_trees):
        # Bootstrap sample the data
        Xb, yb = resample(X_unit, y, replace=True, random_state=rng.randint(0, 2**31 - 1))
        tree = DecisionTreeRegressor(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=rng.randint(0, 2**31 - 1),
        )
        tree.fit(Xb, yb)
        trees.append(tree)
    return trees

# ---------------------------------------
# 3) Acquisition optimization (discrete)
# ---------------------------------------
def propose_via_mc_ei_discrete(model: Model, train_X_t: Tensor, train_y_t: Tensor, bounds_unit_t: Tensor,
                               n_candidates: int = 4096, qmc_samples: int = 512, device=None, dtype=torch.double):
    """
    Vectorized evaluation of MC qEI over a Sobol candidate set in [0,1]^d.
    Returns x_next_unit: Tensor [1, d] in [0,1]^d
    """
    best_f = train_y_t.min().item()
    sampler = SobolQMCNormalSampler(num_samples=qmc_samples)
    acqf = qExpectedImprovement(model=model, best_f=best_f, sampler=sampler)

    d = bounds_unit_t.shape[1]
    cand = draw_sobol_samples(bounds=bounds_unit_t, n=n_candidates, q=1).squeeze(1).to(device=device, dtype=dtype)  # [N, d]
    # Evaluate acquisition for all candidates in one shot: shape [N, 1, d]
    vals = acqf(cand.unsqueeze(1)).detach().cpu().numpy()  # [N]
    idx = int(np.argmax(vals))
    x_next_unit = cand[idx:idx+1, :]                        # [1, d]
    return x_next_unit

# -------------------------------------------------
# 4) Black-box evaluation stub (replace with yours)
# -------------------------------------------------
def evaluate_black_box(x_raw: np.ndarray) -> float:
    """
    Replace this with your real evaluator.
    Input: x_raw shape (2,)
    Return: scalar objective (to minimize)
    """
    raise NotImplementedError("Hook up your real black-box here.")

# -----------------------
# 5) Main optimization
# -----------------------
def main():
    torch.set_default_dtype(torch.double)

    # ---- Load initial data ----
    X_raw = np.load("X_init.npy")     # expected shape [n, 2]
    y_raw = np.load("y_init.npy")     # expected shape [n] or [n, 1]
    if y_raw.ndim > 1:
        y_raw = y_raw.reshape(-1)

    assert X_raw.shape[1] == 2, "Inputs must be 2D."
    n0, d = X_raw.shape

    # ---- Define raw-space bounds ----
    # If you already know the design bounds, SET THEM here:
    # BOUNDS_RAW = np.array([[-5.0, 5.0], [0.0, 10.0]])  # example [[x1_min, x1_max], [x2_min, x2_max]]
    # Otherwise infer from data with a small padding:
    mins_raw, maxs_raw = compute_affine_bounds(X_raw, pad=0.05)
    BOUNDS_RAW = np.vstack([mins_raw, maxs_raw]).T  # shape [d, 2]

    # ---- Normalize inputs to [0,1]^d (common practice for BoTorch) ----
    X_unit = normalize(X_raw, mins_raw, maxs_raw)          # [n, d]

    # ---- Torch tensors ----
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_X_t = torch.tensor(X_unit, dtype=torch.double, device=device)  # [n, d]
    train_y_t = torch.tensor(y_raw, dtype=torch.double, device=device).unsqueeze(-1)  # [n, 1]

    # ---- Fit the tree-ensemble surrogate (explicit hyperparameters) ----
    trees = fit_bootstrapped_cart(
        X_unit=X_unit, y=y_raw,
        n_trees=50,                # ensemble size; increase for smoother uncertainty
        max_depth=5,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=1234,
    )
    model = BootstrappedCARTModel(trees=trees, x_mins=mins_raw, x_maxs=maxs_raw, device=device)

    # ---- Build unit-cube bounds tensor for candidate generation ----
    bounds_unit_t = torch.stack([torch.zeros(d, dtype=torch.double, device=device),
                                 torch.ones(d, dtype=torch.double, device=device)], dim=0)  # [2, d]

    # -----------------------------
    # Iterative BO loop parameters
    # -----------------------------
    N_ITER = 15            # number of BO iterations after the initial data
    N_CAND = 4096          # size of Sobol candidate set for EI maximization
    QMC_SAMPLES = 512      # MC draws for qEI
    history_best = []      # track incumbent best over time

    # Seed incumbent best from the initial data
    incumbent = float(train_y_t.min().item())
    history_best.append(incumbent)

    # -----------------------------
    # BO loop (replace stub to run)
    # -----------------------------
    for it in range(N_ITER):
        # 1) Propose next point in unit space via MC-EI on a Sobol set
        x_next_unit_t = propose_via_mc_ei_discrete(
            model=model,
            train_X_t=train_X_t,
            train_y_t=train_y_t,
            bounds_unit_t=bounds_unit_t,
            n_candidates=N_CAND,
            qmc_samples=QMC_SAMPLES,
            device=device,
            dtype=torch.double,
        )  # [1, d]

        # 2) Convert to raw space for evaluation
        x_next_unit = x_next_unit_t.detach().cpu().numpy()
        x_next_raw = denormalize(x_next_unit, mins_raw, maxs_raw).reshape(-1)  # (d,)

        # 3) Evaluate your black-box (replace stub)
        #    If you cannot evaluate right now, comment these two lines and see the
        #    "One-shot proposal" note at the end to just print x_next_raw.
        y_next = evaluate_black_box(x_next_raw)   # <-- YOUR EVALUATION HERE
        print(f"[Iter {it+1:02d}] Proposed x = {x_next_raw}, f(x) = {y_next:.6f}")

        # 4) Augment training data
        X_raw = np.vstack([X_raw, x_next_raw[None, :]])
        y_raw = np.concatenate([y_raw, [y_next]])
        X_unit = normalize(X_raw, mins_raw, maxs_raw)
        train_X_t = torch.tensor(X_unit, dtype=torch.double, device=device)
        train_y_t = torch.tensor(y_raw, dtype=torch.double, device=device).unsqueeze(-1)

        # 5) Refit the surrogate (lightweight with trees)
        trees = fit_bootstrapped_cart(
            X_unit=X_unit, y=y_raw,
            n_trees=50,
            max_depth=5,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=1234,
        )
        model = BootstrappedCARTModel(trees=trees, x_mins=mins_raw, x_maxs=maxs_raw, device=device)

        # 6) Track convergence
        incumbent = min(incumbent, float(y_next))
        history_best.append(incumbent)

    # -----------------------------
    # Convergence visualization
    # -----------------------------
    iters = np.arange(len(history_best))  # includes initial incumbent at index 0
    plt.figure(figsize=(7, 4))
    plt.plot(iters, history_best, marker="o", lw=2)
    plt.xlabel("Iteration")
    plt.ylabel("Best observed objective")
    plt.title("Bayesian Optimization Convergence (CART surrogate + MC-EI)")
    plt.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

    # Optional: save the final proposal and history
    np.save("bo_history_best.npy", np.array(history_best))
    np.save("bo_X.npy", X_raw)
    np.save("bo_y.npy", y_raw)

if __name__ == "__main__":
    main()